# Get average network map

- run in tmux/ipython, takes ~ 1h I think

In [1]:
import nilearn
import numpy as np
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as op
from brainspace.utils.parcellation import map_to_labels
from infomap import Infomap

from numrisk.fmri_analysis.gradients.utils import get_basic_mask

from fit_assign_consens_plot_nets import threshold_matrix, spatial_filtering, assign_subject_communities_to_reference, get_consensus_assignment



In [2]:
mask, labeling_noParcel = get_basic_mask()
hemi = 'both' 

conn_thresholds = [0.03, 0.04, 0.05, 0.1, 0.2, 0.4] 
conn_thresholds_string = "-".join([str(t) for t in conn_thresholds])


In [3]:
sub = 'All'
bids_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'
source_folder = op.join(bids_folder, 'derivatives', 'correlation_matrices.tryNoHalo') # .parcel
target_folder = op.join(bids_folder,'derivatives','networks_infomap_full')
plot_folder = op.join(bids_folder,'plots_and_ims','networks_infomap_full')


In [10]:
transform_spec = 'tan-hyperbolic' #'_FisherZ' #
confspec= '36Pscrub3BPfilterrunFD104' 

av_cm = np.load(op.join(source_folder,f'cm_av_ses-1_fsav5_unfiltered_confspec-{confspec}{transform_spec}.npy'))
cm_filtered = spatial_filtering(av_cm, bids_folder=bids_folder)

#fn_target_labels_caNets = op.join(target_folder, f'sub-average_consensusMapping_confspec-{confspec}.npy') # mapped to ColeAnticevic nets
#target_labels_caNets = np.load(fn_target_labels_caNets)


In [9]:
# reference mappings
from numrisk.fmri_analysis.gradients.utils import get_glasser_CAatlas_mapping, get_glasser_parcels
mask_glasser, labeling_glasser = get_glasser_parcels(space = 'fsaverage5' )
glasser_CAatlas_mapping, CAatlas_names = get_glasser_CAatlas_mapping()

from brainspace.utils.parcellation import map_to_labels
caNets_fsav5_mapping = map_to_labels(glasser_CAatlas_mapping['ca_network'].values , 
                                     labeling_glasser, mask=mask_glasser, fill=0) #, fill=np.nan) #grad_sub[n_grad-1]
target_labels_caNets = caNets_fsav5_mapping[mask]

In [ ]:
sub_module_mappings_relabelled = []

for conn_threshold in conn_thresholds:
    cm_thresh = threshold_matrix(cm_filtered, proportion=conn_threshold)
    N = cm_thresh.shape[0]
    im = Infomap(preferred_number_of_modules=None) # add flags like '--two-level' if needed
    for i in range(N):
        for j in range(i+1, N):
            w = cm_thresh[i, j]
            if w > 0:
                im.add_link(i, j, w)
    im.run()

    returned_nodes = np.array([node.node_id for node in im.nodes])
    returned_modules = np.array([node.module_id for node in im.nodes])
    full_module_mapping = np.full((N,), -1, dtype=int)  # -1 means unassigned
    full_module_mapping[returned_nodes] = returned_modules

    relabeled_subject, assignments = assign_subject_communities_to_reference(full_module_mapping, target_labels_caNets,  jaccard_threshold=0.1)
    sub_module_mappings_relabelled.append(relabeled_subject)


fn_mappings = op.join(target_folder, f'sub-{sub}_ses-1_task-magjudge_allThresholds_threshs-{conn_thresholds_string}_confspec-{confspec}.npy')
np.save(fn_mappings, np.array(sub_module_mappings_relabelled))

# can happen that small thresholds have very few modules, so we need to check if we have enough
sub_module_mappings_relabelled_sufficient = []
for thresh, mapping in zip(conn_thresholds, sub_module_mappings_relabelled):
    if len(np.unique(mapping)) > 5 or thresh > 0.09:  # More than five unique modules for smal thresholds
        sub_module_mappings_relabelled_sufficient.append(mapping)
    else:
        print(f'Skipping mapping with from threshold {thresh}')

consensus_labels = get_consensus_assignment(sub_module_mappings_relabelled_sufficient)
modules_fsav5 = np.full(mask.shape[0], np.nan, dtype=float)
modules_fsav5[mask] = consensus_labels

fn_consens_mapping = op.join(target_folder, f'sub-{sub}_ses-1_task-magjudge_consensusMapping_threshs-{conn_thresholds_string}_confspec-{confspec}.npy')
np.save(fn_consens_mapping, np.array(consensus_labels))
print(f'Consensus labels saved to {fn_consens_mapping}')